# Phase 4: FLOPs and selected visual comparisons

This notebook completes the remaining Colab work for Phase 4. It does two small jobs:

1. Calculate model complexity for FSRCNN and IMDN using one fixed 256x256 LR input.
2. Reconstruct four predefined test images at x2, x3, and x4 for visual comparison.

It does **not** repeat the full 657-image quality evaluation. FLOPs are reported transparently as `2 x MACs`, because one multiplication plus one addition is counted as two floating-point operations.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/divinesta/SuperResolution-Comparative-Analysis.git'
REPO_ROOT = Path('/content/SuperResolution-Comparative-Analysis')
if REPO_ROOT.exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements.txt')],
    check=True,
)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'thop>=0.1.1'], check=True)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('Repository and dependencies ready.')


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. In Colab, select Runtime > Change runtime type > T4 GPU.')
DEVICE = torch.device('cuda')
DATA_ROOT = Path('/content/drive/MyDrive/FYP_SR_Data')
CHECKPOINT_ROOT = DATA_ROOT / 'checkpoints'
OUTPUT_ROOT = DATA_ROOT / 'results' / 'phase4'
FLOPS_ROOT = OUTPUT_ROOT / 'flops'
VISUAL_ROOT = OUTPUT_ROOT / 'visual_samples'
FIGURE_ROOT = OUTPUT_ROOT / 'visual_figures'
for directory in (FLOPS_ROOT, VISUAL_ROOT, FIGURE_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(DEVICE))
print('Output folder:', OUTPUT_ROOT)


## Part A: model complexity

The same 256x256 LR input size is used for both models and every scale. This keeps the comparison controlled. FSRCNN receives one luminance channel; IMDN receives three RGB channels, matching their real pipelines. This is an operation-count comparison, not a speed measurement.


In [ ]:
from thop import profile
from app.deep_learning.checkpoints import (
    download_official_imdn_checkpoint,
    download_pretrained_fsrcnn_checkpoint,
)
from app.deep_learning.fsrcnn import load_pretrained_fsrcnn
from app.deep_learning.imdn import load_pretrained_imdn
from app.evaluation.experiment import write_results_csv

SCALES = (2, 3, 4)
checkpoint_paths = {
    'fsrcnn': {scale: download_pretrained_fsrcnn_checkpoint(CHECKPOINT_ROOT, scale) for scale in SCALES},
    'imdn': {scale: download_official_imdn_checkpoint(CHECKPOINT_ROOT, scale) for scale in SCALES},
}
print('All six checkpoint files are present and checksum-verified.')


In [ ]:
PROFILE_HEIGHT = 256
PROFILE_WIDTH = 256
complexity_records = []

for method in ('fsrcnn', 'imdn'):
    for scale in SCALES:
        if method == 'fsrcnn':
            model = load_pretrained_fsrcnn(checkpoint_paths[method][scale], scale, DEVICE)
            channels = 1
        else:
            model = load_pretrained_imdn(checkpoint_paths[method][scale], scale, DEVICE)
            channels = 3
        sample = torch.zeros(1, channels, PROFILE_HEIGHT, PROFILE_WIDTH, device=DEVICE)
        with torch.inference_mode():
            macs, parameters = profile(model, inputs=(sample,), verbose=False)
        record = {
            'method': method,
            'scale': f'x{scale}',
            'batch_size': 1,
            'lr_width': PROFILE_WIDTH,
            'lr_height': PROFILE_HEIGHT,
            'input_channels': channels,
            'macs': int(macs),
            'flops': int(2 * macs),
            'gmacs': macs / 1e9,
            'gflops': 2 * macs / 1e9,
            'parameter_count': int(parameters),
            'flop_convention': '2_flops_per_multiply_accumulate',
            'profiler': 'thop',
            'device_used_for_profiling': torch.cuda.get_device_name(DEVICE),
        }
        complexity_records.append(record)
        print(f"{method.upper()} x{scale}: {record['gmacs']:.3f} GMACs, {record['gflops']:.3f} GFLOPs")
        del model, sample
        torch.cuda.empty_cache()

flops_csv = write_results_csv(
    complexity_records, FLOPS_ROOT / 'model_complexity_fixed_256.csv', overwrite=True
)
print('Saved:', flops_csv)


## Part B: selected visual samples

The selection was fixed before viewing the new outputs:

- `Set5/baby.png`: smooth face and gradual colour regions.
- `Set5/butterfly.png`: fine lines and high-frequency texture.
- `Set14/baboon.png`: difficult natural texture.
- `Urban100/img_004.png`: architectural edges and repetitive structure.

Every method receives the same prepared LR image. One reconstruction is produced per method; repeated timing is not performed.


In [ ]:
from app.config import dataset_hr_directory, dataset_lr_directory
from app.deep_learning.alignment import align_reconstruction_to_target
from app.deep_learning.fsrcnn import fsrcnn_upsample
from app.deep_learning.imdn import imdn_upsample
from app.evaluation.images import bicubic_upsample, load_rgb_image, validate_hr_lr_dimensions
from app.traditional.nedi import NEDIConfig, nedi_upsample_rgb

VISUAL_SELECTIONS = (
    ('Set5', 'baby.png', 'smooth face and gradual colour regions'),
    ('Set5', 'butterfly.png', 'fine lines and high-frequency texture'),
    ('Set14', 'baboon.png', 'difficult natural texture'),
    ('Urban100', 'img_004.png', 'architectural edges and repetitive structure'),
)
NEDI_CONFIG = NEDIConfig(window_size=8, edge_threshold=8.0)


In [ ]:
visual_manifest = []

for scale in SCALES:
    fsrcnn_model = load_pretrained_fsrcnn(checkpoint_paths['fsrcnn'][scale], scale, DEVICE)
    imdn_model = load_pretrained_imdn(checkpoint_paths['imdn'][scale], scale, DEVICE)
    for dataset, image_name, reason in VISUAL_SELECTIONS:
        hr_path = dataset_hr_directory(dataset, DATA_ROOT) / image_name
        lr_path = dataset_lr_directory(dataset, scale, DATA_ROOT) / image_name
        reference_hr = load_rgb_image(hr_path)
        lr_image = load_rgb_image(lr_path)
        validate_hr_lr_dimensions(reference_hr, lr_image, scale)

        bicubic = bicubic_upsample(lr_image, reference_hr.size)
        nedi = nedi_upsample_rgb(lr_image, scale, reference_hr.size, NEDI_CONFIG).image
        fsrcnn = align_reconstruction_to_target(
            fsrcnn_upsample(fsrcnn_model, lr_image, DEVICE), reference_hr.size
        ).image
        imdn = align_reconstruction_to_target(
            imdn_upsample(imdn_model, lr_image, DEVICE), reference_hr.size
        ).image

        sample_root = VISUAL_ROOT / dataset / Path(image_name).stem / f'x{scale}'
        sample_root.mkdir(parents=True, exist_ok=True)
        images = {
            'input_lr': lr_image,
            'reference_hr': reference_hr,
            'bicubic': bicubic,
            'nedi': nedi,
            'fsrcnn': fsrcnn,
            'imdn': imdn,
        }
        for label, image in images.items():
            image.save(sample_root / f'{label}.png')
        visual_manifest.append({
            'dataset': dataset,
            'image': image_name,
            'scale': f'x{scale}',
            'selection_reason': reason,
            'hr_width': reference_hr.width,
            'hr_height': reference_hr.height,
            'lr_width': lr_image.width,
            'lr_height': lr_image.height,
            'output_directory': str(sample_root.relative_to(OUTPUT_ROOT)),
        })
        print(f'PASS: {dataset}/{image_name} x{scale}')
    del fsrcnn_model, imdn_model
    torch.cuda.empty_cache()

manifest_csv = write_results_csv(
    visual_manifest, OUTPUT_ROOT / 'visual_selection_manifest.csv', overwrite=True
)
print('Saved:', manifest_csv)


## Part C: full-image comparison sheets

These sheets are for an initial check. After downloading the folder, identical zoom regions will be selected for the final close-up figures.


In [ ]:
import matplotlib.pyplot as plt

DISPLAY_ORDER = ('input_lr', 'reference_hr', 'bicubic', 'nedi', 'fsrcnn', 'imdn')
DISPLAY_TITLES = ('Input LR', 'Reference HR', 'Bicubic', 'NEDI', 'FSRCNN', 'IMDN')

for record in visual_manifest:
    sample_root = OUTPUT_ROOT / record['output_directory']
    figure, axes = plt.subplots(1, 6, figsize=(21, 4))
    for axis, label, title in zip(axes, DISPLAY_ORDER, DISPLAY_TITLES):
        axis.imshow(load_rgb_image(sample_root / f'{label}.png'), interpolation='nearest')
        axis.set_title(title)
        axis.axis('off')
    figure.suptitle(f"{record['dataset']} — {record['image']} — {record['scale']}", fontsize=15)
    figure.tight_layout()
    figure_path = FIGURE_ROOT / f"{record['dataset']}_{Path(record['image']).stem}_{record['scale']}_full.png"
    figure.savefig(figure_path, dpi=180, bbox_inches='tight')
    plt.close(figure)
print(f'Saved {len(visual_manifest)} full-image comparison sheets to {FIGURE_ROOT}.')


In [ ]:
import shutil

archive_path = Path(shutil.make_archive(
    '/content/phase4_remaining_outputs', 'zip', root_dir=OUTPUT_ROOT
))
print('COMPLETE')
print('Drive output:', OUTPUT_ROOT)
print('Download archive:', archive_path)
print('Return phase4_remaining_outputs.zip so the close-up figures and written visual findings can be completed.')


In [ ]:
from google.colab import files
files.download('/content/phase4_remaining_outputs.zip')
